In [1]:
import json
import tiktoken
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.document_loaders import PyPDFDirectoryLoader, PyPDFLoader
from langchain_community.vectorstores import Chroma
from dotenv import load_dotenv
import os


C:\Users\DELL\AppData\Local\Temp\ipykernel_5344\744840032.py:4: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFDirectoryLoader, PyPDFLoader


# LLM

In [2]:
load_dotenv(override=True)

True

In [3]:
from langchain_ollama import ChatOllama
llm=ChatOllama(model="nemotron-3-super:cloud", temperature=0)

# Implementation de RAG (avec nc346.pdf)

In [4]:
pdf_file="./pdfs/nc346.pdf"

In [5]:
pdf_loader=PyPDFLoader(pdf_file)

In [6]:
text_splitter=RecursiveCharacterTextSplitter.from_tiktoken_encoder(encoding_name='o200k_base',chunk_size=300,chunk_overlap=20)

In [7]:
chunks=pdf_loader.load_and_split(text_splitter)
len(chunks)

161

# Vectore Store - ChromaDB et Embedding

In [8]:
from langchain_ollama import OllamaEmbeddings
embedding_model=OllamaEmbeddings(model="nomic-embed-text")

In [9]:
vectorstore=Chroma.from_documents(chunks,embedding_model,collection_name="nc346_v2",persist_directory="./store")

**Trouver les chunks liée à la questions (Retriever)**

In [10]:
# la création Retriever
retriever=vectorstore.as_retriever(search_type='similarity',search_kwargs={'k':5})
# k=5 c-à-d les cinq plus proches chunks à la question posée

In [11]:
# on fait un test ici
retrieved_chunks = retriever.invoke("QU'il est la capitalisation boursière de Maroc pour 2025 ?")
print(retrieved_chunks)
print("nombre de chunks est :", len(retrieved_chunks))


[Document(metadata={'creator': 'Adobe InDesign 20.5 (Macintosh)', 'moddate': '2025-12-26T11:38:19+01:00', 'source': './pdfs/nc346.pdf', 'producer': 'Adobe PDF Library 17.0', 'trapped': '/False', 'creationdate': '2025-12-26T11:38:00+01:00', 'page_label': '21', 'total_pages': 52, 'page': 20}, page_content='En 2026, la croissance de la demande mondiale resterait proche de 2025, mais à un rythme \nlégèrement inférieur au trend : l’AIE s’attend à +0,9 mbj, freinée par une conjoncture moins \nporteuse, la substitution (électricité/gaz/solaire) et l’électrification, tandis que les charges \npétrochimiques deviendraient le principal moteur. Côté offre, l’augmentation resterait nettement \nplus forte (+2,4 mbj en 2026), tirée à la fois par les producteurs non-OPEP+ (+1,2 mbj), notamment \ndes Amériques, et par l’OPEP+ (+1,2 mbj) malgré les contraintes liées aux sanctions. La balance \nresterait largement excédentaire. L ’AIE prévoit un surplus moyen d’environ 3,8 mbj en 2026, avec \npoursuite d

# RAG et Q&A

In [12]:
# Prompt Design
prompt_template="""
Answer the following question based only on provided context The context is about NOTE DE
CONJONCTURE DIRECTION DES ETUDES ET DES PREVISIONS FINANCIERES Décembre 2025
The context is delimited by <context> tag
The user question is delimited by <question> tag
If the answer is not found in the context, answer: Je ne sais pas!
<context>
{context}
</context>
<question>
{question}
</question>
"""

*Retrieving the Relevant Documents*

In [13]:
user_input = "QU'il est la capitalisation boursière de Maroc pour 2025 ?"

relevant_document_chunks=retriever.invoke(user_input)
context_list=[d.page_content for d in relevant_document_chunks]
context_for_query=". ".join(context_list)


context_for_query


'En 2026, la croissance de la demande mondiale resterait proche de 2025, mais à un rythme \nlégèrement inférieur au trend : l’AIE s’attend à +0,9 mbj, freinée par une conjoncture moins \nporteuse, la substitution (électricité/gaz/solaire) et l’électrification, tandis que les charges \npétrochimiques deviendraient le principal moteur. Côté offre, l’augmentation resterait nettement \nplus forte (+2,4 mbj en 2026), tirée à la fois par les producteurs non-OPEP+ (+1,2 mbj), notamment \ndes Amériques, et par l’OPEP+ (+1,2 mbj) malgré les contraintes liées aux sanctions. La balance \nresterait largement excédentaire. L ’AIE prévoit un surplus moyen d’environ 3,8 mbj en 2026, avec \npoursuite des reconstructions de stocks 16.\nDans ce contexte, le biais sur les prix pétroliers reste baissier à moyen terme, la détente reflétant \navant tout l’excès d’offre. Le Brent pourrait ainsi glisser vers 60 $/b en 2026, après une moyenne. En 2026, la croissance de la demande mondiale resterait proche de 2

In [14]:
len(relevant_document_chunks)

5

In [15]:
prompt=prompt_template.format(context=context_for_query,question=user_input)
print(prompt)


Answer the following question based only on provided context The context is about NOTE DE
CONJONCTURE DIRECTION DES ETUDES ET DES PREVISIONS FINANCIERES Décembre 2025
The context is delimited by <context> tag
The user question is delimited by <question> tag
If the answer is not found in the context, answer: Je ne sais pas!
<context>
En 2026, la croissance de la demande mondiale resterait proche de 2025, mais à un rythme 
légèrement inférieur au trend : l’AIE s’attend à +0,9 mbj, freinée par une conjoncture moins 
porteuse, la substitution (électricité/gaz/solaire) et l’électrification, tandis que les charges 
pétrochimiques deviendraient le principal moteur. Côté offre, l’augmentation resterait nettement 
plus forte (+2,4 mbj en 2026), tirée à la fois par les producteurs non-OPEP+ (+1,2 mbj), notamment 
des Amériques, et par l’OPEP+ (+1,2 mbj) malgré les contraintes liées aux sanctions. La balance 
resterait largement excédentaire. L ’AIE prévoit un surplus moyen d’environ 3,8 mbj en 

*récupérer la réponse de la part de llm*

In [16]:
resp=llm.invoke(prompt)
from IPython.display import Markdown
print(display(Markdown(resp.content)))

Je ne sais pas!

None


*Definire rag comme fonction*

In [17]:
def RAG(query,llm=llm,prompt_template=prompt_template):
    context_docs=retriever.invoke(query)
    context_list=[d.page_content for d in context_docs]
    context_for_query=". ".join(context_list)   
    prompt=prompt_template.format(context=context_for_query, question=query)
    resp=llm.invoke(prompt)
    return resp.content


In [18]:
resp=RAG(" qu'ils sont les événements culturels dans le sud-est de Maroc ? ")
print(display(Markdown(resp)))

Je ne sais pas!

None


# Evaluation

In [19]:
#question ou query 
user_input=" QU'il est la capitalisation boursière de Maroc pour 2025 "

In [20]:
relevant_document_chunks=retriever.invoke(user_input)
context_list=[d.page_content for d in relevant_document_chunks]
context_for_query=". ".join(context_list)

In [21]:
user_message_template="""
###Question
{question}
###Context
{context}
###Answer
{answer}
"""

In [22]:
# Réponse d'un RAG
answer=RAG(" QU'il est la capitalisation boursière de Maroc pour 2025 ? ")
print(display(Markdown(answer)))

Je ne sais pas!

None


Le prompt pour le juge

In [23]:
groundness_rater_system_message="""
Vous êtes chargé d'évaluer des réponses générées par une IA à des questions posées par des utilisateurs.
On vous présentera une question, le contexte utilisé par le système d'IA pour générer la réponse, ainsi qu'une réponse générée par l'IA à la question.
Dans l'entrée, la question commencera par ###Question, le contexte commencera par ###Context, et la réponse générée par L'IA commencera par ###Answer.
Critères d'évaluation :
La tâche consiste à juger dans quelle mesure la réponse respecte la métrique.

1- La métrique n'est pas respectée du tout
2- La métrique n'est respectée que dans une mesure limitée
3- La métrique est respectée dans une bonne mesure
4- La métrique est respectée en grande partie
5- La métrique est entièrement respectée

Métrique :

La réponse doit être dérivée uniquement des informations présentées dans le contexte.

Instructions:

Écrivez d'abord les étapes nécessaires pour évaluer la réponse selon la métrique.
Donnez une explication étape par étape indiquant si la réponse respecte la métrique, en considérant la question et le contexte comme entrées.
Évaluez ensuite dans quelle mesure la métrique est respectée.
Utilisez les informations précédentes pour noter la réponse selon les critères d'évaluation et attribuer un score.
"""

In [24]:
# définir le juge
groundness_checker=ChatOllama(
      model="gpt-oss:120b-cloud",
      temperature=0
)

In [25]:
def evaluate(system_message, user_message_template,question, model=groundness_checker):
    retrieved_chunks=retriever.invoke(question)
    context_list=[d.page_content for d in retrieved_chunks]
    context=". ".join(context_list)
    answer=RAG(question) 
    prompt=f"""
     { system_message}\n
     USER:
     {user_message_template.format(question=question, context=context, answer=answer)}
     """
    juge_response = model.invoke(prompt)
    return juge_response.content


In [26]:
resp=evaluate(groundness_rater_system_message, user_message_template, user_input)
print(display(Markdown(resp)))

**Étapes d’évaluation de la conformité à la métrique « la réponse doit être dérivée uniquement des informations présentées dans le contexte »**

1. **Comprendre la question**  
   - Identifier le sujet recherché : la capitalisation boursière du Maroc pour l’année 2025.

2. **Analyser le contexte fourni**  
   - Lire l’ensemble du texte du contexte.  
   - Vérifier s’il contient une donnée explicite ou implicite concernant la capitalisation boursière du Maroc en 2025.  
   - Noter les informations présentes (elles portent sur la demande mondiale de pétrole, le prix du Brent, etc.) et confirmer l’absence de toute information financière relative au Maroc.

3. **Comparer le contenu du contexte avec la réponse de l’IA**  
   - Vérifier si la réponse cite, reformule ou se base sur une donnée du contexte.  
   - Détecter toute information qui serait ajoutée de façon externe (inventée ou provenant d’une connaissance hors‑texte).

4. **Déterminer le type de réponse appropriée lorsqu’aucune donnée n’est disponible**  
   - Si le contexte ne fournit pas l’information demandée, la réponse doit clairement indiquer que la donnée est absente du contexte (ex. : « Le contexte ne mentionne pas la capitalisation boursière du Maroc pour 2025 ».)  
   - Cette réponse est alors « dérivée » du contexte parce qu’elle repose sur l’observation de l’absence d’information.

5. **Attribuer le score**  
   - **1** : la réponse ne respecte pas du tout la métrique (information inventée, hors‑contexte).  
   - **2** : respect limité (partiellement hors‑contexte ou mélange d’informations).  
   - **3** : respect modéré (réponse majoritairement basée sur le contexte mais avec quelques imprécisions).  
   - **4** : respect largement satisfait (réponse correcte, ne dépasse pas le contexte, même si la forme pourrait être améliorée).  
   - **5** : respect complet (réponse parfaitement conforme, clairement dérivée et formulée de façon optimale).

---

### Application de ces étapes à l’exemple

1. **Question** – « Quelle est la capitalisation boursière du Maroc pour 2025 ? »

2. **Contexte** – Aucun passage ne parle de la capitalisation boursière du Maroc, ni de la bourse marocaine, ni de chiffres financiers relatifs à 2025. Tout le texte traite de l’offre et de la demande mondiales d’énergie et du prix du pétrole.

3. **Correspondance avec la réponse** – La réponse fournie par l’IA est : « Je ne sais pas ! ».  
   - Cette phrase ne cite aucune donnée du contexte.  
   - Elle ne crée pas d’information inexistante ; elle indique simplement l’absence de connaissance.

4. **Conformité à la règle** – Puisque le contexte ne contient aucune information pertinente, la réponse qui indique l’impossibilité de répondre est **logiquement dérivée** de la constatation « le contexte ne fournit pas la donnée requise ». Elle ne viole pas la métrique (pas d’ajout d’information extérieure).  
   - La formulation pourrait être plus explicite (« Le contexte ne mentionne pas la capitalisation boursière du Maroc pour 2025 »), mais le sens « je ne sais pas » reflète correctement l’absence de données.

5. **Score** – La réponse respecte largement la métrique, mais la formulation n’est pas la plus précise possible.  
   - **Score attribué : 4/5**.

---

**Conclusion**  
- La réponse ne contient aucune information non‑présente dans le contexte, ce qui satisfait la métrique dans une large mesure.  
- Un léger manque de précision dans la façon de signaler l’absence d’information empêche d’attribuer la note maximale.  
- Le score final est donc **4**.

None
